In [1]:
import importlib
from tqdm.auto import tqdm
import os
import pickle
import sys
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

parent_dir = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'analysis' else Path(os.getcwd())
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))


import torch
import transformers
from transformers import AutoTokenizer, AutoModel
import numpy as np 
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import pandas as pd

from safetensors.torch import load_model
from model import GenderChunkedClassifier

from eval_utils.viz_utils import plot_error_bar, plot_violins
from eval_utils.inference_utils import load_generated_output, find_threshold_for_N, calculate_metrics
from gender_dataset import MultiSpeciesGenderDataChunkedDataset

from scipy.stats import mannwhitneyu
from scipy.stats import kstest

/disk/10tb/home/chepurova/miniconda3/envs/dna-lm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
! ls 'mammals_inference_runs/human_model_runs/'

 human_model_evaluation_evaluation_results.jsonl
'mammals_model_runs_HUMAN_CONTIGS_human_contigs_16x3072_bs_128_lr_1e-05_chrY_from_checkpoint_run_1_checkpoint-50500_model.safetensors_train_species_Apodemus sylvaticus_force_Y_sampling_False_Y_ratio_None_X_ratio_None_60000_per_sample_142.pckl'
'mammals_model_runs_HUMAN_CONTIGS_human_contigs_16x3072_bs_128_lr_1e-05_chrY_from_checkpoint_run_1_checkpoint-50500_model.safetensors_train_species_Arvicanthis niloticus_force_Y_sampling_False_Y_ratio_None_X_ratio_None_60000_per_sample_142.pckl'
'mammals_model_runs_HUMAN_CONTIGS_human_contigs_16x3072_bs_128_lr_1e-05_chrY_from_checkpoint_run_1_checkpoint-50500_model.safetensors_train_species_Bos javanicus_force_Y_sampling_False_Y_ratio_None_X_ratio_None_60000_per_sample_142.pckl'
'mammals_model_runs_HUMAN_CONTIGS_human_contigs_16x3072_bs_128_lr_1e-05_chrY_from_checkpoint_run_1_checkpoint-50500_model.safetensors_train_species_Bos taurus_force_Y_sampling_False_Y_ratio_None_X_ratio_None_60000_per_sampl

In [3]:
file_cnt = 0
folder_path = 'mammals_inference_runs/human_model_runs/'
for file in os.listdir(folder_path):
    if 'pckl' in file:
        file_cnt += 1

In [4]:
file_cnt

21

In [5]:
folder_path = 'mammals_inference_runs/human_model_runs/'
for file in os.listdir(folder_path):
    if 'pckl' in file:

        test_dump = os.path.join(folder_path, file)

        d = pickle.load(open(test_dump, 'rb'))
        test_sample2probs, test_sample2labels, test_sample2chromosomes = d['sample_ids_probs'], d['sample_ids_labels'], d['sample_ids_sampled_chromosomes']
        

        species = list(test_sample2probs.keys())[0]
        new_test_sample2probs = {species+"_male": [], species+"_female": [] }
        new_test_sample2chromosomes = {species+"_male": [], species+"_female": [] }
        new_test_sample2labels = {species+"_male": 0, species+"_female": 1}


        for i, elem in enumerate(test_sample2probs[species]):
            if test_sample2labels[species][i] == 0:
                new_test_sample2probs[species+"_male"].append(elem)
                new_test_sample2chromosomes[species+"_male"].append(test_sample2chromosomes[species][i])

            elif test_sample2labels[species][i] == 1:
                new_test_sample2probs[species+"_female"].append(elem)
                new_test_sample2chromosomes[species+"_female"].append(test_sample2chromosomes[species][i])

        for key in new_test_sample2probs.keys():
            new_test_sample2probs[key] = np.array(new_test_sample2probs[key]).reshape(-1, 1)

        print(species)
        print(mannwhitneyu(new_test_sample2probs[species+'_male'], new_test_sample2probs[species+'_female'], alternative='two-sided').pvalue[0])
        print(kstest(new_test_sample2probs[species+'_male'], new_test_sample2probs[species+'_female'], alternative='two-sided').pvalue[0])
        print("---"*10)


Lutra lutra
0.6979012425104552
0.7404549820891746
------------------------------
Ovis aries
1.6949623058735502e-06
0.00017883927298252924
------------------------------
Homo sapiens
1.2772862291186544e-14
8.966182385040244e-08
------------------------------
Mustela lutreola
1.834909409003304e-06
7.750479572379197e-05
------------------------------
Pan paniscus
9.65711171441901e-05
0.0069965801125259665
------------------------------
Bos javanicus
4.955874420846041e-06
7.766470224484682e-06
------------------------------
Globicephala melas
5.086320372902907e-05
0.00230966255602783
------------------------------
Papio anubis
3.868552121355754e-09
6.5826894887536565e-06
------------------------------
Arvicanthis niloticus
0.00035922434302971515
0.004411726553014644
------------------------------
Mustela erminea
2.451846746298885e-06
0.00012008407947517156
------------------------------
Mus musculus
7.5213049214407e-05
3.2872020002798097e-05
------------------------------
Budorcas taxicolo

In [5]:
test_sample2probs

{'Lutra lutra': array([[0.49609375],
        [0.49609375],
        [0.47460938],
        ...,
        [0.49804688],
        [0.49609375],
        [0.4765625 ]], dtype=float32)}